In [ ]:
# result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")))

# str(result)
# # Convert data types to match BigQuery schema
# result[, id_series := as.integer(id_series)]
# result[, id_pin := as.integer(id_pin)]
# result[, date_adm := as.Date(date_adm, format = "%m/%d/%Y")]
# result[, time_adm := as.ITime(time_adm)]
# result[is.na(time_adm), time_adm := as.ITime("00:00:00")]
# result[, date_dis := as.Date(date_dis, format = "%m/%d/%Y")]
# result[, time_dis := as.ITime(time_dis)]
# result[is.na(time_dis), time_dis := as.ITime("00:00:00")]
# result[, date_rec := as.Date(date_rec, format = "%m/%d/%Y")]
# result[, date_ref := as.Date(date_ref, format = "%m/%d/%Y")]
# result[, date_check := as.Date(date_check, format = "%m/%d/%Y")]
# result[, id_hci := as.character(id_hci)]

# # Convert character "0"/"1" to logical for Boolean fields
# result[, clin_outpatient := as.logical(as.integer(clin_outpatient))]
# result[, clin_emergency := as.logical(as.integer(clin_emergency))]

# result[, pat_type := as.character(pat_type)]
# result[, clin_acc := as.character(clin_acc)]
# result[, pat_rel := as.character(pat_rel)]
# result[, pat_bdate := as.Date(pat_bdate, format = "%m/%d/%Y")]
# result[, pat_age := as.numeric(pat_age)]
# result[, pat_age_orig := as.numeric(pat_age)]
# result[, pat_sex := as.character(pat_sex)]
# result[, pat_bwt := as.numeric(pat_bwt)]
# result[, pat_memcat_parent := as.character(pat_memcat_parent)]
# result[, pat_memcat_child := as.character(pat_memcat_child)]
# result[, clin_discharge := as.integer(clin_discharge)]

# result[, claim_status := as.character(claim_status)]
# result[, claim_payout := as.numeric(claim_payout)]
# result[, claim_charge := as.numeric(claim_charge)]
# result[, date_ext := as.Date(date_ext, format = "%m/%d/%Y")]
# result[, id_year := as.integer(id_year)]

# result[, c1_orig := as.character(c1_orig)]
# result[, c2_orig := as.character(c2_orig)]

# result[, pdx := as.character(pdx)]
# result[, pdx_code := as.integer(pdx_code)]

# result[, `:=`(
#   # c1 = ifelse(c1 == pdx, NA_character_, c1), # Keep c1 even if it's the pdx
#   # c2 = ifelse(c2 == pdx, NA_character_, c2), # Keep c2 even if it's the pdx
#   clin_icd = mapply(function(pdx_var, sdx_var) sdx_var[sdx_var != pdx_var], pdx, clin_icd, SIMPLIFY = FALSE)
# )]

# saveRDS(result, here(checkpoint_6_path, paste0(checkpoint_6_prefix, ".rds")), compress = FALSE)

# result_dt <- data.table::copy(result)

# # # Convert columns to Date objects, ignoring NA values
# date_columns <- names(result_dt)[grepl("date", names(result_dt))]
# # result_dt[, pat_bdate_orig := pat_bdate]
# # result_dt[, (date_columns) := lapply(.SD, as.Date, format = "%Y-%m-%d"), .SDcols = date_columns]

# # Create a data.table for rows with negative `pat_age`
# negative_age_dt <- result_dt[pat_age < 0, ]

# # Create a data.table for rows with non-negative `pat_age`
# positive_age_dt <- result_dt[pat_age >= 0, ]

# # Process rows with non-negative `pat_age`
# positive_age_dt[, pat_bdate := dmy(generate_dob(format(pat_bdate, "%Y-%m-%d"), pat_age, format(date_adm, "%Y-%m-%d")))]

# # For negative `pat_age`, keep `pat_bdate` as it is
# # negative_age_dt[, pat_bdate := pat_bdate_orig]

# # Combine the processed data back together
# result_dt <- rbind(positive_age_dt, negative_age_dt)

# # Calculate 'pat_age' only for rows where 'pat_bdate' is not missing
# result_dt[!is.na(pat_bdate), pat_age := floor(as.numeric(interval(pat_bdate, date_adm) / years(1)))]

# # Function to split a list column by '||' and ensure a fixed number of columns
# split_codes <- function(dt, column, prefix, max_cols) {
#   # Apply strsplit to each element in the list column
#   if (is_unix) {
#     split_list <- mclapply(dt[[column]], function(x) unlist(strsplit(x, "\\|")))
#     # Ensure that each list element has exactly max_cols elements
#     split_cols <- mclapply(1:max_cols, function(i) sapply(split_list, function(x) if (length(x) >= i) x[[i]] else NA_character_))
#   } else {
#     split_list <- future_lapply(dt[[column]], function(x) unlist(strsplit(x, "\\|")))
#     # Ensure that each list element has exactly max_cols elements
#     split_cols <- future_lapply(1:max_cols, function(i) sapply(split_list, function(x) if (length(x) >= i) x[[i]] else NA_character_))
#   }

#   # Convert to data.table
#   split_dt <- as.data.table(split_cols)
#   # Name the columns appropriately
#   setnames(split_dt, paste0(prefix, 1:max_cols))
#   return(split_dt)
# }

# # Apply the function to clin_icd and icd9_list
# sdx_columns <- split_codes(result_dt, "clin_icd", "sdx", 12)
# proc_columns <- split_codes(result_dt, "icd9_list", "proc", 20)

# if (is_unix) {
#   proc_columns[, (names(proc_columns)) := mclapply(.SD, as.character)]
#   sdx_columns[, (names(sdx_columns)) := mclapply(.SD, as.character)]
# } else {
#   proc_columns[, (names(proc_columns)) := future_lapply(.SD, as.character)]
#   sdx_columns[, (names(sdx_columns)) := future_lapply(.SD, as.character)]
# }


# # Combine the split columns back into result_dt
# result_dt <- cbind(result_dt, sdx_columns, proc_columns)

# # Rename columns
# setnames(result_dt,
#   old = c("clin_discharge", "pat_bwt", "pat_age", "pat_sex"),
#   new = c("discharge", "birthweight", "patage", "patsex")
# )

# # Replace NA with "None" in non-date columns
# non_date_columns <- setdiff(names(result_dt), date_columns)
# if (is_unix) {
#   result_dt[, (non_date_columns) := mclapply(.SD, function(x) ifelse(is.na(x), NA_character_, x)), .SDcols = non_date_columns]
# } else {
#   result_dt[, (non_date_columns) := future_lapply(.SD, function(x) ifelse(is.na(x), NA_character_, x)), .SDcols = non_date_columns]
# }

# # Columns you want to come first
# priority_columns <- c(
#   "id_series", "id_pin", "date_adm", "time_adm", "date_dis", "time_dis",
#   "date_rec", "date_ref", "date_check", "id_hci", "id_hcp", "clin_outpatient",
#   "clin_emergency", "pat_type", "clin_acc", "pat_rel", "pat_bdate", "patage",
#   "patsex", "birthweight", "pat_memcat_parent", "pat_memcat_child", "discharge",
#   "c1", "c2", "claim_status", "claim_payout", "claim_charge",
#   "date_ext", "id_year", "clin_icd", "clin_rvs", "c1_orig",
#   "c2_orig", "icd9_list", "pdx", "pdx_code"
# )

# # Remaining columns
# remaining_columns <- c(
#   "sdx1", "sdx2", "sdx3", "sdx4", "sdx5", "sdx6",
#   "sdx7", "sdx8", "sdx9", "sdx10", "sdx11", "sdx12",
#   "proc1", "proc2", "proc3", "proc4", "proc5", "proc6",
#   "proc7", "proc8", "proc9", "proc10", "proc11", "proc12",
#   "proc13", "proc14", "proc15", "proc16", "proc17", "proc18",
#   "proc19", "proc20" # , "ageday" , "discharge", "birthweight"
# )

# # Combine the lists, ensuring no duplicates
# desired_columns <- unique(c(priority_columns, remaining_columns))

# # Reorder the data.table columns
# setcolorder(result_dt, desired_columns)

# # Add missing columns with "None" values if they are not already in the data.table
# missing_columns <- setdiff(desired_columns, names(result_dt))
# result_dt[, (missing_columns) := NA_character_]

# # Initialize the 'ageday' column with NA
# result_dt[, ageday := NA_real_]

# # Calculate 'ageday' only for valid rows where pat_age < 1 and date_adm/pat_bdate are non-NA
# result_dt[
#   !is.na(patage) & patage < 1 & !is.na(date_adm) & !is.na(pat_bdate),
#   ageday := as.numeric(difftime(date_adm, pat_bdate, units = "days"))
# ]

# # Convert time_adm to numeric
# result_dt[, time_adm := as.numeric(time_adm)]
# result_dt[, time_dis := as.numeric(time_dis)] # Also do the same for time_dis if needed

# # Convert time from seconds since midnight to "HH:MM:SS" format
# result_dt[, time_adm := sprintf("%02d:%02d:%02d", time_adm %/% 3600, (time_adm %% 3600) %/% 60, time_adm %% 60)]
# result_dt[, time_dis := sprintf("%02d:%02d:%02d", time_dis %/% 3600, (time_dis %% 3600) %/% 60, time_dis %% 60)]

# # Convert ITime to character format "HH:MM:SS"
# result_dt[, time_adm := format(as.ITime(time_adm), "%H:%M:%S")]
# result_dt[, time_dis := format(as.ITime(time_dis), "%H:%M:%S")]

# # Combine Date and Time and convert to POSIXct
# result_dt[, date_adm := as.character(paste(date_adm, time_adm), format = "%Y-%m-%d %H:%M:%S")]
# result_dt[, date_dis := as.character(paste(date_dis, time_dis), format = "%Y-%m-%d %H:%M:%S")]

# # Replace NA values in 'ageday' with "None"
# result_dt[, ageday := fifelse(is.na(ageday), NA, as.character(ageday))]

# before_replacing_with_none <- data.table::copy(result_dt)

# replace_result <- replace_empty_with_na(result_dt) # TODO: See if this works if changed from replace_empty_with_none

# result_dt <- replace_result$return_data

# # Write the final DataFrame to CSV
# saveRDS(result_dt, here(checkpoint_7_path, paste0(checkpoint_7a_prefix, suffix, ".rds")), compress = FALSE)

# test <- data.table::copy(result_dt)

# if (is_unix) {
#   test[, names(test) := mclapply(.SD, function(col) {
#     if (is.character(col)) {
#       return(iconv(col, from = "", to = "UTF-8"))
#     } else {
#       return(col)
#     }
#   })]
# } else {
#   test[, names(test) := future_lapply(.SD, function(col) {
#     if (is.character(col)) {
#       return(iconv(col, from = "", to = "UTF-8"))
#     } else {
#       return(col)
#     }
#   })]
# }

# if (to_debug) print(head(test, 10))

# for_fwrite <- test[, c(
#   "id_series", "date_adm", "date_dis", "patage", "patsex", "discharge", "pdx",
#   paste0("sdx", 1:12), paste0("proc", 1:20), "birthweight", "ageday"
# ), with = FALSE]

# # saveRDS(as.data.frame(test), here(checkpoint_7_path, paste0(checkpoint_7b_prefix, suffix, ".rds")), compress = FALSE)
# fwrite(for_fwrite, here(checkpoint_7_path, paste0(checkpoint_7b_prefix, suffix, ".csv")))

# if (to_debug) for (col in date_columns) print(unique(result_dt[[col]]))

# # pandas <- import("pandas")

# if (to_debug) print(sapply(test, class))


In [ ]:
# # Use reticulate to run the following Python code within the R environment
# py_run_string(paste0("
# import pandas as pd
# import numpy as np
# from io import StringIO
# import sys


# # Read the CSV with the specified dtype
# pandas_df = pd.read_csv('", csv_path, "',

# dtype = {
#     'id_series': 'int64',
#     # 'id_pin': 'int64',
#     'date_adm': 'string',  # POSIXct/POSIXt in R
#     # 'time_adm': 'string',  # character in R
#     'date_dis': 'string',  # POSIXct/POSIXt in R
#     # 'time_dis': 'string',  # character in R
#     # 'date_rec': 'string',  # Date in R
#     # 'date_ref': 'string',  # Date in R
#     # 'date_check': 'string',  # Date in R
#     # 'id_hci': 'string',
#     # 'id_hcp': 'string',
#     # 'clin_outpatient': 'string',
#     # 'clin_emergency': 'string',
#     # 'pat_type': 'string',
#     # 'clin_acc': 'string',
#     # 'pat_rel': 'string',
#     # 'pat_bdate': 'string',  # Date in R
#     'patage': 'float64',  # numeric in R
#     'patsex': 'string',
#     'birthweight': 'float64',  # character in R
#     # 'pat_memcat_parent': 'string',
#     # 'pat_memcat_child': 'string',
#     'discharge': 'Int64',  # character in R
#     # 'c1': 'string',
#     # 'c2': 'string',
#     # 'claim_status': 'string',
#     # 'claim_payout': 'string',  # character in R
#     # 'claim_charge': 'string',  # character in R
#     # 'date_ext': 'string',  # Date in R
#     # 'id_year': 'int64',  # integer in R
#     # 'clin_icd': 'object',  # list in R
#     # 'clin_rvs': 'string',
#     # 'c1_orig': 'string',
#     # 'c2_orig': 'string',
#     # 'icd9_list': 'string',
#     'pdx': 'string',
#     # 'pdx_code': 'int64',  # integer in R
#     'sdx1': 'string',
#     'sdx2': 'string',
#     'sdx3': 'string',
#     'sdx4': 'string',
#     'sdx5': 'string',
#     'sdx6': 'string',
#     'sdx7': 'string',
#     'sdx8': 'string',
#     'sdx9': 'string',
#     'sdx10': 'string',
#     'sdx11': 'string',
#     'sdx12': 'string',
#     'proc1': 'string',
#     'proc2': 'string',
#     'proc3': 'string',
#     'proc4': 'string',
#     'proc5': 'string',
#     'proc6': 'string',
#     'proc7': 'string',
#     'proc8': 'string',
#     'proc9': 'string',
#     'proc10': 'string',
#     'proc11': 'string',
#     'proc12': 'string',
#     'proc13': 'string',
#     'proc14': 'string',
#     'proc15': 'string',
#     'proc16': 'string',
#     'proc17': 'string',
#     'proc18': 'string',
#     'proc19': 'string',
#     'proc20': 'string',
#     'ageday': 'float64'
# }
# # , parse_dates = [
# #     'date_adm',
# #     'date_dis',
# #     'date_rec',
# #     'date_ref',
# #     'date_check',
# #     'pat_bdate',
# #     'date_ext'
# # ]
# )

# pandas_df = pandas_df.replace(pd.NA, None)
# pandas_df = pandas_df.replace(np.nan, None)
# pandas_df = pandas_df.replace('<NA>', None)
# pandas_df = pandas_df.replace('None', None)

# # Capture pandas_df.info() output
# buffer = StringIO()
# pandas_df.info(buf=buffer)
# info_output = buffer.getvalue()

# info_output
# "))

# # Print the captured output in R
# cat(py$info_output)

In [ ]:
# # if (to_debug) str(py$output)

# # # Convert data types to match BigQuery schema
# # result <- as.data.table(py$output)

# # # Apply format_id function to each column in parallel or sequentially
# # columns_to_format <- c("id_series")

# # # Define the format_id function
# # format_id <- function(x) {
# #   x <- as.character(x)
# #   integer_x <- suppressWarnings(as.integer(x))
# #   x <- trimws(formatC(integer_x, format = "f", digits = 0))
# #   x[x == "NA" | is.na(integer_x)] <- NA_character_
# #   x
# # }

# # # Apply format_id to each column safely
# # if (is_unix) {
# #   # Make a copy of the columns to avoid directly accessing the data.table object in parallel
# #   formatted_cols <- mclapply(columns_to_format, function(col) {
# #     column_data <- result[[col]] # Extract column data outside the parallel loop
# #     return(format_id(column_data))
# #   }, mc.cores = parallel::detectCores())
# # } else {
# #   formatted_cols <- lapply(columns_to_format, function(col) {
# #     column_data <- result[[col]] # Extract column data
# #     return(format_id(column_data))
# #   })
# # }

# # # Assign the formatted results back to the respective columns
# # for (i in seq_along(columns_to_format)) {
# #   result[[columns_to_format[i]]] <- formatted_cols[[i]]
# # }

# # # full_data <- data.table::copy(test)

# # # Define the function to process data with parallelization using mclapply where possible
# # process_data_parallel <- function(data) {
# #   # Process character columns: Replace 'None' and '<NA>' with NA
# #   char_cols <- names(data)[sapply(data, is.character)]
# #   data[, (char_cols) := mclapply(.SD, function(col) {
# #     col[col == "None" | col == "<NA>"] <- NA_character_
# #     return(col)
# #   }), .SDcols = char_cols]

# #   # Process numeric columns: Replace NaN with NA
# #   num_cols <- names(data)[sapply(data, is.numeric)]
# #   data[, (num_cols) := mclapply(.SD, function(col) {
# #     col[is.nan(col)] <- NA_real_
# #     return(col)
# #   }), .SDcols = num_cols]

# #   # Process list columns
# #   list_cols <- names(data)[sapply(data, is.list)]
# #   data[, (list_cols) := mclapply(.SD, function(col) {
# #     lapply(col, function(x) {
# #       if (is.character(x)) {
# #         x[x == "None" | x == "<NA>"] <- NA_character_
# #       }
# #       return(x)
# #     })
# #   }), .SDcols = list_cols]

# #   return(data)
# # }

# # # Apply the parallelized function to both 'full_data' and 'result'
# # # full_data <- process_data_parallel(full_data)
# # result <- process_data_parallel(result)

# # # setnames(full_data,
# # #   old = c("discharge", "birthweight", "patage", "patsex"),
# # #   new = c("clin_discharge", "pat_bwt", "pat_age", "pat_sex")
# # # )

# # # full_data[, (c(paste0("sdx", 1:12), paste0("proc", 1:20))) := NULL]

# # # result <- merge(full_data, result, by = "id_series", all = TRUE)

# # date_columns <- c(
# #   "date_adm", "date_dis", "date_rec", "date_ref", "date_check",
# #   "pat_bdate", "date_ext"
# # )

# # time_columns <- c("time_adm", "time_dis")

# # if (is_unix) {
# #   result[, (date_columns) := mclapply(.SD, as.Date), .SDcols = date_columns]
# #   result[, (time_columns) := mclapply(.SD, as.ITime), .SDcols = time_columns]
# # } else {
# #   result[, (date_columns) := future_lapply(.SD, as.Date), .SDcols = date_columns]
# #   result[, (time_columns) := future_lapply(.SD, as.ITime), .SDcols = time_columns]
# # }

# # # Convert string columns to arrays
# # array_columns <- c("id_hcp")

# # if (is_unix) {
# #   result[, (array_columns) := mclapply(.SD, function(x) strsplit(x, "\\|\\|")), .SDcols = array_columns]

# #   # Replace NULL (empty) arrays with an empty character vector
# #   result[, (array_columns) := mclapply(.SD, function(x) {
# #     lapply(
# #       x,
# #       function(y) if (length(y) == 0 || is.null(y) || all(is.na(y))) character(0) else y
# #     )
# #   }), .SDcols = array_columns]
# # } else {
# #   result[, (array_columns) := future_lapply(.SD, function(x) strsplit(x, "\\|\\|")), .SDcols = array_columns]

# #   # Replace NULL (empty) arrays with an empty character vector
# #   result[, (array_columns) := future_lapply(.SD, function(x) {
# #     lapply(
# #       x,
# #       function(y) if (length(y) == 0 || is.null(y) || all(is.na(y))) character(0) else y
# #     )
# #   }), .SDcols = array_columns]
# # }

# # # # Convert string columns to arrays
# # # array_columns <- c("c1", "c2", "clin_icd", "clin_rvs")
# # # result[, (array_columns) := lapply(.SD, function(x) strsplit(x, "\\|")), .SDcols = array_columns]

# # # # Replace NULL (empty) arrays with an empty character vector
# # # result[, (array_columns) := lapply(.SD, function(x) {
# # #   lapply(
# # #     x,
# # #     function(y) if (length(y) == 0 || is.null(y) || all(is.na(y))) character(0) else y
# # #   )
# # # }), .SDcols = array_columns]

# # # Manual fixes
# # result[, clin_rvs := icd9_list]

# # if (is_unix) {
# #   result[, pat_bwt := as.character(unlist(mclapply(pat_bwt, function(pat_bwt) as.numeric(ifelse(is.null(pat_bwt), NA_real_, pat_bwt)))))]
# #   result[, ageday := as.character(unlist(mclapply(ageday, function(ageday) as.numeric(ifelse(is.null(ageday), NA_real_, ageday)))))]
# # } else {
# #   result[, pat_bwt := as.character(unlist(future_lapply(pat_bwt, function(pat_bwt) as.numeric(ifelse(is.null(pat_bwt), NA_real_, pat_bwt)))))]
# #   result[, ageday := as.character(unlist(future_lapply(ageday, function(ageday) as.numeric(ifelse(is.null(ageday), NA_real_, ageday)))))]
# # }

# # if (is_unix) {
# #   # Convert 'warning_code' list column to a simple character column using mclapply
# #   result[, warning_code := mclapply(warning_code, function(x) {
# #     if (is.null(x) || length(x) == 0) {
# #       return(NA_character_) # Set NA for NULL or empty lists
# #     } else {
# #       return(paste(unlist(x, recursive = TRUE), collapse = "|")) # Flatten the list and join with "|"
# #     }
# #   })] # Adjust mc.cores to your system's capacity

# #   # Convert 'error_code' list column to a simple character column using mclapply
# #   result[, error_code := mclapply(error_code, function(x) {
# #     if (is.null(x) || length(x) == 0) {
# #       return(NA_character_) # Set NA for NULL or empty lists
# #     } else {
# #       return(paste(unlist(x, recursive = TRUE), collapse = "|")) # Flatten the list and join with "|"
# #     }
# #   })]

# #   # Convert 'c1' list column using mclapply
# #   result[, c1 := mclapply(c1, function(x) {
# #     if (is.null(x) || length(x) == 0) {
# #       return(NA_character_) # Set NA for NULL or empty lists
# #     } else {
# #       return(as.character(unlist(x)))
# #     }
# #   })]

# #   # Convert 'c2' list column using mclapply
# #   result[, c2 := mclapply(c2, function(x) {
# #     if (is.null(x) || length(x) == 0) {
# #       return(NA_character_) # Set NA for NULL or empty lists
# #     } else {
# #       return(as.character(unlist(x)))
# #     }
# #   })]
# # } else {
# #   # Convert 'warning_code' list column to a simple character column using sapply
# #   result[, warning_code := sapply(warning_code, function(x) {
# #     if (is.null(x) || length(x) == 0) {
# #       return(NA_character_) # Set NA for NULL or empty lists
# #     } else {
# #       return(paste(unlist(x, recursive = TRUE), collapse = "|")) # Flatten the list and join with "|"
# #     }
# #   })]

# #   # Convert 'error_code' list column to a simple character column using sapply
# #   result[, error_code := sapply(error_code, function(x) {
# #     if (is.null(x) || length(x) == 0) {
# #       return(NA_character_) # Set NA for NULL or empty lists
# #     } else {
# #       return(paste(unlist(x, recursive = TRUE), collapse = "|")) # Flatten the list and join with "|"
# #     }
# #   })]

# #   # Convert 'c1' list column using sapply
# #   result[, c1 := sapply(c1, function(x) {
# #     if (is.null(x) || length(x) == 0) {
# #       return(NA_character_) # Set NA for NULL or empty lists
# #     } else {
# #       return(as.character(unlist(x)))
# #     }
# #   })]

# #   # Convert 'c2' list column using sapply
# #   result[, c2 := sapply(c2, function(x) {
# #     if (is.null(x) || length(x) == 0) {
# #       return(NA_character_) # Set NA for NULL or empty lists
# #     } else {
# #       return(as.character(unlist(x)))
# #     }
# #   })]
# # }


# # # Ensure the final columns are of type character and no longer lists
# # result[, warning_code := as.character(warning_code)]
# # result[, error_code := as.character(error_code)]
# # result[, c1 := as.character(c1)]
# # result[, c2 := as.character(c2)]
# # # str(result)

# # Convert string columns to arrays (list of character vectors)
# array_columns <- c("warning_code", "error_code")

# if (is_unix) {
#   result[, (array_columns) := mclapply(.SD, function(x) strsplit(x, "\\|")), .SDcols = array_columns]
#   # Replace NULL (empty) arrays with an empty character vector
#   result[, (array_columns) := mclapply(.SD, function(x) {
#     lapply(x, function(y) {
#       # Ensure y is not a list and apply condition checks safely
#       if (is.character(y) && all(!is.na(y)) && all(y == "NA")) {
#         return(character(0))
#       }
#       # Check for other conditions: NULL, empty list, or all NA
#       if (length(y) == 0 || is.null(y) || all(is.na(y))) {
#         return(character(0))
#       } else {
#         return(y)
#       }
#     })
#   }), .SDcols = array_columns]
# } else {
#   result[, (array_columns) := future_lapply(.SD, function(x) strsplit(x, "\\|")), .SDcols = array_columns]
#   # Replace NULL (empty) arrays with an empty character vector
#   result[, (array_columns) := future_lapply(.SD, function(x) {
#     lapply(x, function(y) {
#       # Ensure y is not a list and apply condition checks safely
#       if (is.character(y) && all(!is.na(y)) && all(y == "NA")) {
#         return(character(0))
#       }
#       # Check for other conditions: NULL, empty list, or all NA
#       if (length(y) == 0 || is.null(y) || all(is.na(y))) {
#         return(character(0))
#       } else {
#         return(y)
#       }
#     })
#   }), .SDcols = array_columns]
# }

# # result[, id_series := as.character(id_series)]
# # result[, id_pin := as.character(id_pin)]
# # # rename columns for thai grouper and bq push, dropping the pre-renamed source columns, also drop mdc and dc
# # result[, clin_pdx := pdx]
# # result[, clin_sdx := clin_icd]
# # result[, pdx := NULL]
# # result[, clin_icd := NULL]
# # result[, py_pdc := pdc]
# # result[, py_pccl := pccl]
# # result[, py_warn := warning_code]
# # result[, py_err := error_code]
# # result[, pdc := NULL]
# # result[, pccl := NULL]
# # result[, warning_code := NULL]
# # result[, error_code := NULL]
# # result[, mdc := NULL]
